# Debug Harmony dual1 — inj-wrap duals + stack peek

**Not a Submit.** `DUAL_SCREEN_REPS = 1`. Sister notebook uses the other depth.

Flow: find best 1x → wrap duals with that inj → screen all duals ×1 → confirm → gated stack peek → farm.

## Steps
1. Competition + `gpt-oss` GGUF, GPU ON
2. Internet ON for llama_cpp install cell, then OK OFF
3. Run All → refresh Output for `attack_run_summary.txt`
Compare `selected=` / dual fire / `stack_peeked` vs the dual2 notebook.


In [ ]:
import sys, os, glob, time
from pathlib import Path

sys.argv = [sys.argv[0]]

def log(msg: str) -> None:
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line, flush=True)
    try:
        p = Path('/kaggle/working/debug_heartbeat.txt')
        p.parent.mkdir(parents=True, exist_ok=True)
        with p.open('a', encoding='utf-8') as f:
            f.write(line + '\n')
    except Exception:
        pass

log('path probe')
dataset_root = None
for candidate in glob.glob('/kaggle/input/**/kaggle_evaluation', recursive=True):
    dataset_root = str(Path(candidate).parent)
    if dataset_root not in sys.path:
        sys.path.insert(0, dataset_root)
    break
log(f'dataset_root={dataset_root}')
assert dataset_root, 'Attach competition input'


In [ ]:
ATTACK_B64 = (
    "..."
)

from pathlib import Path
import base64, time

def log(msg: str) -> None:
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line, flush=True)
    try:
        with Path('/kaggle/working/debug_heartbeat.txt').open('a', encoding='utf-8') as f:
            f.write(line + '\n')
    except Exception:
        pass

attack_bytes = base64.b64decode(ATTACK_B64)
out = Path('/kaggle/working/attack.py')
out.write_bytes(attack_bytes)
log(f'wrote {out} ({len(attack_bytes)} bytes)')


In [ ]:
import glob, os, time
from pathlib import Path

def log(msg: str) -> None:
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line, flush=True)
    try:
        with Path('/kaggle/working/debug_heartbeat.txt').open('a', encoding='utf-8') as f:
            f.write(line + '\n')
    except Exception:
        pass

AGENT_MODE = 'gpt_oss'
BUDGET_S = 600.0
MAX_TOOL_HOPS = 8
N_CTX = 4096
N_GPU_LAYERS = -1
DEBUG_SHRINK_RESERVE = True
DUAL_SCREEN_REPS = 1

def pick_gguf(*needles: str):
    hits = []
    for p in glob.glob('/kaggle/input/**/*.gguf', recursive=True):
        low = p.lower()
        if all(n.lower() in low for n in needles):
            hits.append(p)
    if not hits and needles:
        for p in glob.glob('/kaggle/input/**/*.gguf', recursive=True):
            if needles[0].lower() in p.lower():
                hits.append(p)
    if not hits:
        return None
    hits.sort(key=lambda p: (Path(p).stat().st_size, p))
    return hits[0]

GPT_OSS_GGUF = pick_gguf('gpt-oss') or pick_gguf('gpt_oss') or pick_gguf('oss-20')
log(f'AGENT_MODE={AGENT_MODE} BUDGET_S={BUDGET_S} DUAL_SCREEN_REPS={DUAL_SCREEN_REPS}')
log(f'GPT_OSS_GGUF={GPT_OSS_GGUF}')
if AGENT_MODE == 'gpt_oss' and not GPT_OSS_GGUF:
    raise RuntimeError('no GGUF — attach gpt-oss model')


In [ ]:
import sys, subprocess, time

def log(msg: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

if AGENT_MODE != 'gpt_oss':
    log('skip llama_cpp install')
else:
    try:
        from llama_cpp import Llama  # noqa: F401
        log('llama_cpp already installed')
    except ModuleNotFoundError:
        log('installing llama-cpp-python…')
        cmds = [
            [sys.executable, '-m', 'pip', 'install', '-q', 'llama-cpp-python',
             '--extra-index-url', 'https://abetlen.github.io/llama-cpp-python/whl/cu121'],
            [sys.executable, '-m', 'pip', 'install', '-q', 'llama-cpp-python',
             '--extra-index-url', 'https://abetlen.github.io/llama-cpp-python/whl/cu122'],
            [sys.executable, '-m', 'pip', 'install', '-q', 'llama-cpp-python'],
        ]
        last_err = None
        for cmd in cmds:
            try:
                subprocess.check_call(cmd)
                from llama_cpp import Llama  # noqa: F401
                log('llama_cpp installed OK')
                break
            except Exception as e:
                last_err = e
                log(f'attempt failed: {type(e).__name__}')
        else:
            raise RuntimeError('Could not install llama_cpp — Internet ON?') from last_err
    log('llama_cpp OK')


In [ ]:
import importlib.util
import time
from pathlib import Path

import aicomp_sdk
from aicomp_sdk import AttackRunConfig
from aicomp_sdk.agents import build_agent
from aicomp_sdk.core.env.sandbox import SandboxEnv
from aicomp_sdk.guardrails.optimal import Guardrail as OptimalGuardrail

def log(msg: str) -> None:
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line, flush=True)
    try:
        with Path('/kaggle/working/debug_heartbeat.txt').open('a', encoding='utf-8') as f:
            f.write(line + '\n')
    except Exception:
        pass


def load_attack():
    path = Path('/kaggle/working/attack.py')
    spec = importlib.util.spec_from_file_location('kaggle_debug_attack', path)
    mod = importlib.util.module_from_spec(spec)
    import sys
    sys.modules[spec.name] = mod
    spec.loader.exec_module(mod)
    mod.DUAL_SCREEN_REPS = int(DUAL_SCREEN_REPS)
    if DEBUG_SHRINK_RESERVE:
        mod.MARGIN_S = 20.0
        mod.MAX_DYNAMIC_RESERVE_S = 60.0
        log('debug: shrunk reserve + DUAL_SCREEN_REPS=%s' % mod.DUAL_SCREEN_REPS)
    return mod


def make_agent():
    if AGENT_MODE == 'deterministic':
        return build_agent('deterministic')
    from aicomp_sdk.agents.gpt_oss_agent import (
        GPTOSSAgent, build_gpt_oss_backend_config, build_gpt_oss_parser,
    )
    from aicomp_sdk.agents.hf_chat_template.backends.llama_cpp import LlamaCppChatTemplateBackend
    log(f'loading llama_cpp gpt_oss from {GPT_OSS_GGUF}')
    t0 = time.time()
    cfg = build_gpt_oss_backend_config(model_path=GPT_OSS_GGUF, local_files_only=True)
    backend = LlamaCppChatTemplateBackend.from_model_path(
        model_path=GPT_OSS_GGUF, config=cfg, n_ctx=N_CTX, n_gpu_layers=N_GPU_LAYERS, verbose=False,
    )
    agent = GPTOSSAgent(backend, parser=build_gpt_oss_parser(None, model_id=cfg.model_id or 'openai/gpt-oss-20b'))
    log(f'loaded gpt_oss in {time.time()-t0:.1f}s')
    return agent

mod = load_attack()
summary = Path('/kaggle/working/attack_run_summary.txt')
mod.SUMMARY_PATHS = (summary, Path('attack_run_summary.txt'))
fixtures = Path(aicomp_sdk.__file__).resolve().parent / 'fixtures'
env = SandboxEnv(seed=1, fixtures_dir=fixtures, agent=make_agent(), guardrail=OptimalGuardrail())
log(f'running AttackAlgorithm budget={BUDGET_S}s dual_screen={DUAL_SCREEN_REPS}')
t0 = time.time()
findings = mod.AttackAlgorithm().run(env, AttackRunConfig(time_budget_s=BUDGET_S, max_tool_hops=MAX_TOOL_HOPS))
log(f'done wall_s={time.time()-t0:.1f} findings={len(findings) if findings else 0}')


In [ ]:
from pathlib import Path
import time

summary = Path('/kaggle/working/attack_run_summary.txt')
print('\n======== attack_run_summary.txt ========\n', flush=True)
if summary.exists():
    print(summary.read_text(encoding='utf-8'), flush=True)
else:
    print('MISSING summary', flush=True)
print(f'findings={len(findings) if findings else 0}', flush=True)
print('Compare dual1 vs dual2: selected= stack_peeked= dual fire/exact', flush=True)
